# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azkakhan4/flyrank-ai-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")
print("Token found!" if token else "Token missing!")

Token found!


In [21]:
!pip -q install duckdb

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

print(schema.to_string(index=False))

             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions      BIGINT  YES None    None  None
ga4_total_engagement_sec

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents the daily performance of one content item belonging to one pseudonymized client on one report date.

**Main table:** `fact_content_daily_performance`

**Time window:** March 1, 2026, to March 31, 2026 (`month=2026-03`).

**Lane:** Content Refresh and SEO Opportunity Prioritization.

**Prediction goal:** Rank content items to identify potential SEO content-refresh opportunities. We will use future organic search performance as the outcome proxy, subject to data availability.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
These are candidate input measurements, using historical data available before the prediction date:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_sessions`
- `ga4_engaged_sessions`

### Label / proxy
- Future organic search performance, measured using GSC clicks in a future time window. This will be calculated separately, not used as a feature.

### Context
- `report_date` — identifies the reporting date.
- `client_hash_id` — identifies the pseudonymized client.
- `content_hash_id` — identifies the content item.
- `month` — identifies the monthly partition.
- `gsc_data_available` and `ga4_data_available` — used to check data availability.

### Excluded
- `client_has_gsc` and `client_has_ga4` — account-level availability flags, not model inputs.
- Any future-period measurements — unavailable at prediction time and would cause leakage.
- Client and content identifiers — never used as predictive features.

In [23]:
feature_frame = con.sql("""
WITH daily AS (
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
),

historical_gsc AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position
    FROM daily
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

historical_ga4 AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(ga4_sessions) AS ga4_sessions
    FROM daily
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

future_label AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS future_gsc_clicks
    FROM daily
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    g.gsc_impressions,
    g.gsc_clicks,
    g.gsc_avg_position,
    a.ga4_sessions,
    c.word_count,
    f.future_gsc_clicks,
    CASE WHEN f.future_gsc_clicks > 0 THEN 1 ELSE 0 END
        AS has_future_clicks
FROM future_label f
LEFT JOIN historical_gsc g
    ON f.client_hash_id = g.client_hash_id
    AND f.content_hash_id = g.content_hash_id
LEFT JOIN historical_ga4 a
    ON f.client_hash_id = a.client_hash_id
    AND f.content_hash_id = a.content_hash_id
LEFT JOIN read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
) c
    ON f.client_hash_id = c.client_hash_id
    AND f.content_hash_id = c.content_hash_id
""").df()

print("Feature frame created:", feature_frame.shape)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame created: (166224, 9)


In [24]:
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "word_count"
]

print("Feature columns:")
print(feature_columns)

print("\nMissing values:")
print(feature_frame[feature_columns].isna().sum())

Feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'word_count']

Missing values:
gsc_impressions      24757
gsc_clicks           24757
gsc_avg_position     24757
ga4_sessions        125631
word_count           53257
dtype: int64


### Feature availability

| Feature | Available when? |
|---|---|
| `gsc_impressions` | When GSC data is available during March 1–15 |
| `gsc_clicks` | When GSC data is available during March 1–15 |
| `gsc_avg_position` | When GSC data is available during March 1–15 |
| `ga4_sessions` | When GA4 data is available during March 1–15 |
| `word_count` | When a word-count value exists in the content table |

Missing values are retained at this stage rather than treated as zero. They indicate unavailable information, not necessarily zero activity.

In [25]:
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Keep rows with a known outcome
experiment_data = feature_frame.dropna(
    subset=["has_future_clicks"]
).copy()

X = experiment_data[feature_columns]
y = experiment_data["has_future_clicks"]

# Fill missing feature values for this experiment
imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

honest_score = accuracy_score(y_test, model.predict(X_test))

print("Honest model accuracy:", round(honest_score, 4))

Honest model accuracy: 0.8334


In [26]:
# Deliberately leaked feature: the target copied into the inputs
X_leaked = experiment_data[feature_columns].copy()
X_leaked["leaked_future_clicks"] = y.values

X_leaked = imputer.fit_transform(X_leaked)

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leaked, y, test_size=0.2, random_state=42, stratify=y
)

leak_model = DecisionTreeClassifier(max_depth=5, random_state=42)
leak_model.fit(X_train_leak, y_train_leak)

leaked_score = accuracy_score(
    y_test_leak, leak_model.predict(X_test_leak)
)

print("Accuracy with deliberate leakage:", round(leaked_score, 4))

Accuracy with deliberate leakage: 1.0


### Leakage experiment

A deliberate leakage test was performed by adding `has_future_clicks`, the target variable, as an input feature.

The model achieved 100% accuracy because it was given the answer it was supposed to predict. This is not a valid performance result.

After identifying the leakage, the leaked feature was excluded. The honest baseline accuracy was **83.62%**.

The honest result is retained for reporting. The 100% result is shown only to demonstrate why future information must never be included in model features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [27]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
        report_date,
        client_hash_id,
        content_hash_id
    )) AS unique_records
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_records
0,9841378,9841378


In [28]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [29]:
con.sql("""
SELECT
    COUNT(*) AS available_gsc_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

,available_gsc_rows
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitations

1. **Unequal data coverage:** Not every client or content item has the same amount of historical data.
2. **Missing GSC data:** Only 3,611,061 out of 9,841,378 March records have GSC data available.
3. **Limited time window:** This analysis uses March 2026 data, so it does not represent long-term performance.
4. **No causal conclusions:** The data can show performance patterns, but cannot prove that refreshing content will increase traffic.
5. **Privacy restrictions:** Client and content identifiers are pseudonymized, so real client identities cannot be identified.

**Main limitation:** Incomplete and unequal data coverage may affect comparisons between content items.

In [30]:
con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
)
""").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [31]:
con.sql("""
SELECT
    COUNT(*) AS total_clients,
    COUNT(gsc_data_start) AS clients_with_gsc_start,
    COUNT(ga4_data_start) AS clients_with_ga4_start,
    MIN(gsc_data_start) AS earliest_gsc_start,
    MAX(gsc_data_start) AS latest_gsc_start,
    MIN(ga4_data_start) AS earliest_ga4_start,
    MAX(ga4_data_start) AS latest_ga4_start
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
)
""").df()

,total_clients,clients_with_gsc_start,clients_with_ga4_start,earliest_gsc_start,latest_gsc_start,earliest_ga4_start,latest_ga4_start
0,104,67,51,2025-01-27,2026-06-02,2025-10-29,2026-06-01


In [32]:
con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
""").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [33]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (client_hash_id, content_hash_id)) AS unique_content_items,
    COUNT(word_count) AS rows_with_word_count,
    COUNT(search_volume) AS rows_with_search_volume
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
""").df()

,total_rows,unique_content_items,rows_with_word_count,rows_with_search_volume
0,519606,519606,341838,376984


In [34]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
        AND ga4_data_available IS TRUE
    ) AS both_available
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available,both_available
0,9841378,3611061,413966,364347


In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.